# ImageDETECTIVE: fakes from four more open models (Kaggle, 13 Nov 2026)

More generator **decoder families** for the whole-image head, from the same 792 pair captions as before
(every fake has a real photo with the same content). All logic is in `scripts/generate_open_models.py`.

| model | licence | decoder | use |
|---|---|---|---|
| Sana 1600M 1024px | Apache 2.0 | DC-AE | training |
| Wuerstchen v2 | MIT | Paella VQGAN | training |
| aMUSEd-512 | OpenRAIL++ | VQ-VAE (token model) | training or the unseen test |
| Kolors | Apache 2.0 | SDXL's VAE | **dev only** (SDXL stays unseen) |

**Settings before running** (right panel): *Accelerator*: **GPU T4 x2**, *Internet*: **on**. No token needed (none is gated).

**Two steps:** first run all cells with `LIMIT = 3` (quick test, ~10 min incl. downloads). If the images look right and
no model shows `black`, set `LIMIT = None`, then **Save Version → Save & Run All (Commit)**. The images end up in the
version's output: `open_gen/<model>/`. A model that fails is skipped; the others go on.

In [ ]:
LIMIT = 3  # quick test: 3 images per model. Full run: None

In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/hf"  # model downloads stay out of the notebook output
!rm -rf /tmp/repo && git clone --depth 1 https://github.com/phantom007xy/ImageDETECTIVE.git /tmp/repo
# Kaggle's preinstalled torchao is too old for diffusers 0.41 (no FqnToConfig) and crashes its import;
# we use no quantization, so without torchao diffusers simply skips that part.
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0 sentencepiece
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import subprocess
import sys

OUT = "/kaggle/working/open_gen"
os.makedirs(OUT, exist_ok=True)
limit = f" --limit {LIMIT}" if LIMIT is not None else ""


def queue(models, device):
    """One shell command per GPU: the models one after the other (a failing model does not stop the next)."""
    cmds = [f"{sys.executable} scripts/generate_open_models.py --model {m} --prompts data/prompts/pairs_v1_prompts.csv "
            f"data/prompts/pairs_v2_prompts.csv --out-dir {OUT} --device {device}{limit} > {OUT}/{m}.log 2>&1" for m in models]
    return subprocess.Popen("; ".join(cmds), shell=True, cwd="/tmp/repo")


jobs = [queue(("sana", "amused"), "cuda:0"), queue(("wuerstchen", "kolors"), "cuda:1")]
for j in jobs:
    j.wait()
for m in ("sana", "amused", "wuerstchen", "kolors"):
    print(f"===== {m}")
    !tail -n 4 {OUT}/{m}.log

In [ ]:
import csv
import glob
from collections import Counter

from PIL import Image

for model in ("sana", "amused", "wuerstchen", "kolors"):
    path = f"{OUT}/{model}/manifest.csv"
    rows = list(csv.DictReader(open(path))) if os.path.exists(path) else []
    seconds = [float(r["seconds"]) for r in rows]
    print(model, dict(Counter(r["status"] for r in rows)),
          f"| {sum(seconds) / max(1, len(seconds)):.1f} s per image" if seconds else "")
    for p in sorted(glob.glob(f"{OUT}/{model}/*.png"))[:2]:  # a few images to look at
        im = Image.open(p)
        display(im.resize((384, round(384 * im.height / im.width))))